# Лабораторная работа № 1. Разведочный анализ больших данных
Студент: Жакупов М.С, группа ЭВМ 2.2
Датасет: US Accidents (2016–2023), https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents
## Часть 1. Загрузка и подготовка данных

In [2]:
import os
from pyspark.sql import SparkSession, functions as F

USER = os.getenv('USER')
BASE = f'hdfs:///user/{USER}/zhakupov_directory'
CSV_PATH = f'{BASE}/us_accidents/US_Accidents_March23.csv'
WAREHOUSE = f'{BASE}/warehouse'
DB = 'zhakupov_database'

spark = (SparkSession.builder
    .appName('zhakupov_lab1_us_accidents')
    .master('yarn')
    .config('spark.submit.deployMode', 'client')
    .config('spark.executor.instances', '2')
    .config('spark.executor.cores', '2')
    .config('spark.executor.memory', '4g')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.spark_catalog.type', 'hadoop')
    .config('spark.sql.catalog.spark_catalog.warehouse', WAREHOUSE)
    .getOrCreate())

print('Spark', spark.version, '| приложение', spark.sparkContext.applicationId)

:: loading settings :: url = jar:file:/opt/spark-3.5.2-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/user5/.ivy2/cache
The jars for the packages stored in: /home/user5/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-529842a5-beb5-4df8-b3f8-7ccfcb13dd5f;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 in central
:: resolution report :: resolve 598ms :: artifacts dl 19ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spa

Spark 3.5.2 | приложение application_1794419134879_0100


In [3]:
raw = (spark.read
       .option('header', True)
       .option('quote', '"')
       .option('escape', '"')
       .option('mode', 'FAILFAST')
       .csv(CSV_PATH))

print('Столбцов:', len(raw.columns))
print('Строк:', raw.count())
raw.printSchema()
raw.show(2, vertical=True, truncate=False)

Столбцов: 46


Строк: 7728394
root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: string (nullable = true)
 |-- Start_Time: string (nullable = true)
 |-- End_Time: string (nullable = true)
 |-- Start_Lat: string (nullable = true)
 |-- Start_Lng: string (nullable = true)
 |-- End_Lat: string (nullable = true)
 |-- End_Lng: string (nullable = true)
 |-- Distance(mi): string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: string (nullable = true)
 |-- Temperature(F): string (nullable = true)
 |-- Wind_Chill(F): string (nullable = true)
 |-- Humidity(%): string (nullable = true)
 |-- Pressure(in): string (nullable = true)
 

26/11/24 17:00:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


-RECORD 0------------------------------------------------------------------------------------------------------
 ID                    | A-1                                                                                   
 Source                | Source2                                                                               
 Severity              | 3                                                                                     
 Start_Time            | 2016-02-08 05:46:00                                                                   
 End_Time              | 2016-02-08 11:00:00                                                                   
 Start_Lat             | 39.865147                                                                             
 Start_Lng             | -84.058723                                                                            
 End_Lat               | NULL                                                                           

### Типы признаков и их преобразование
Все поля прочитаны как строки. Имена столбцов приводим к виду `distance_mi`, `humidity_pct`,
затем приводим типы: `severity` — int; координаты, расстояние и погода — double;
время — timestamp; признаки объектов дороги — boolean; остальное — строки.

Время в источнике записано в трёх форматах: без долей секунды, с 6 и с 9 знаками после точки
(вторые два встречаются только у `Source1`). Каждый формат разбирается своим шаблоном.
Для проверки считаем непустые значения, которые не удалось преобразовать.

In [7]:
import re
spark.conf.set('spark.sql.ansi.enabled', 'false')
spark.conf.set('spark.sql.legacy.timeParserPolicy', 'CORRECTED')

def snake(name):
    name = name.replace('(%)', '_pct')
    return re.sub(r'\((\w+)\)', r'_\1', name).lower()

src = raw.toDF(*[snake(c) for c in raw.columns])

int_cols = ['severity']
double_cols = ['start_lat', 'start_lng', 'end_lat', 'end_lng', 'distance_mi',
               'temperature_f', 'wind_chill_f', 'humidity_pct', 'pressure_in',
               'visibility_mi', 'wind_speed_mph', 'precipitation_in']
time_cols = ['start_time', 'end_time', 'weather_timestamp']
bool_cols = ['amenity', 'bump', 'crossing', 'give_way', 'junction', 'no_exit', 'railway',
             'roundabout', 'station', 'stop', 'traffic_calming', 'traffic_signal', 'turning_loop']

# Какие форматы времени встречаются (каждая цифра заменена на 9)
(src.groupBy(F.regexp_replace('start_time', r'\d', '9').alias('start_time_format'), 'source')
    .count().show(truncate=False))

TS_FORMATS = ['yyyy-MM-dd HH:mm:ss',
              'yyyy-MM-dd HH:mm:ss.SSSSSS',
              'yyyy-MM-dd HH:mm:ss.SSSSSSSSS']

def convert(c):
    x = F.trim(F.col(c))
    x = F.when(x == '', None).otherwise(x)
    if c in int_cols:    return x.cast('int')
    if c in double_cols: return x.cast('double')
    if c in time_cols:   return F.coalesce(*[F.to_timestamp(x, f) for f in TS_FORMATS])
    if c in bool_cols:   return x.cast('boolean')
    return x

typed = src.select(*[convert(c).alias(c) for c in src.columns])

# Ошибки преобразования: исходное значение было, а после приведения стало NULL
checked = int_cols + double_cols + time_cols + bool_cols
failed = src.agg(*[
    F.sum(F.when((F.trim(src[c]) != '') & convert(c).isNull(), 1).otherwise(0)).alias(c)
    for c in checked]).first().asDict()
print('Ошибок преобразования:', sum(failed.values()))
typed.printSchema()

+-----------------------------+-------+-------+
|start_time_format            |source |count  |
+-----------------------------+-------+-------+
|9999-99-99 99:99:99          |Source3|97389  |
|9999-99-99 99:99:99          |Source2|3305373|
|9999-99-99 99:99:99          |Source1|3582466|
|9999-99-99 99:99:99.999999999|Source1|682322 |
|9999-99-99 99:99:99.999999   |Source1|60844  |
+-----------------------------+-------+-------+



Ошибок преобразования: 0
root
 |-- id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- severity: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- distance_mi: double (nullable = true)
 |-- description: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- airport_code: string (nullable = true)
 |-- weather_timestamp: timestamp (nullable = true)
 |-- temperature_f: double (nullable = true)
 |-- wind_chill_f: double (nullable = true)
 |-- humidity_pct: double (nullable = true)
 |-- pressure_in: double (n

### Производные признаки и сохранение в Apache Iceberg
Добавляем признаки для анализа: длительность ДТП в минутах, год, месяц, день недели, час и дату начала.
Время в данных — местное для штата (столбец `timezone`), поэтому час и день недели отражают местное время.
Подготовленная таблица сохраняется в Iceberg; исходный CSV в HDFS не изменяется.

In [9]:
df = (typed
    .withColumn('duration_min',
                (F.col('end_time').cast('long') - F.col('start_time').cast('long')) / 60)
    .withColumn('start_date',  F.to_date('start_time'))
    .withColumn('start_year',  F.year('start_time'))
    .withColumn('start_month', F.month('start_time'))
    .withColumn('start_dow',   F.dayofweek('start_time'))      # 1 = воскресенье ... 7 = суббота
    .withColumn('start_hour',  F.hour('start_time')))

df.select('start_time', 'end_time', 'duration_min', 'start_year',
          'start_month', 'start_dow', 'start_hour').show(5)

TABLE = f'spark_catalog.{DB}.us_accidents'
spark.sql(f'CREATE NAMESPACE IF NOT EXISTS spark_catalog.{DB}')
df.writeTo(TABLE).using('iceberg').createOrReplace()

saved = spark.table(TABLE)
print('Таблица:', TABLE)
print('Строк:', saved.count(), '| столбцов:', len(saved.columns))
spark.sql(f'SHOW TABLES IN spark_catalog.{DB}').show(truncate=False)

+-------------------+-------------------+------------+----------+-----------+---------+----------+
|         start_time|           end_time|duration_min|start_year|start_month|start_dow|start_hour|
+-------------------+-------------------+------------+----------+-----------+---------+----------+
|2016-02-08 05:46:00|2016-02-08 11:00:00|       314.0|      2016|          2|        2|         5|
|2016-02-08 06:07:59|2016-02-08 06:37:59|        30.0|      2016|          2|        2|         6|
|2016-02-08 06:49:27|2016-02-08 07:19:27|        30.0|      2016|          2|        2|         6|
|2016-02-08 07:23:34|2016-02-08 07:53:34|        30.0|      2016|          2|        2|         7|
|2016-02-08 07:39:07|2016-02-08 08:09:07|        30.0|      2016|          2|        2|         7|
+-------------------+-------------------+------------+----------+-----------+---------+----------+
only showing top 5 rows



Таблица: spark_catalog.zhakupov_database.us_accidents
Строк: 7728394 | столбцов: 52
+-----------------+------------+-----------+
|namespace        |tableName   |isTemporary|
+-----------------+------------+-----------+
|zhakupov_database|us_accidents|false      |
+-----------------+------------+-----------+



In [10]:
spark.stop()